# 04 - Cruce con autoadscripcion indigena (Conindig) y Sierra Tarahumara
Hipotesis: los municipios con mayor recurrencia en tasas altas (Bocoyna,
Balleza, Guerrero, Guachochi, Carichi) pertenecen a la Sierra Tarahumara,
region con alta poblacion rarahumara/rarámuri, y esto podria explicar parte
del patron observado en 03_recurrence_analysis.ipynb.

IMPORTANTE - limitacion metodologica: Conindig es autoadscripcion INDIVIDUAL
de cada caso de suicidio (si la persona fallecida se identificaba como
indigena), NO es el % de poblacion indigena del municipio (eso requeriria
Censo/Conteo INEGI, no lo tenemos en este notebook). Son dos cosas distintas:
aqui solo podemos decir 'que % de los CASOS de suicidio en la sierra eran
personas que se autoidentificaron indigenas', no 'la poblacion indigena
tiene mayor riesgo de suicidio' (eso necesitaria el denominador correcto).

Definicion de Sierra Tarahumara (17 municipios) tomada de fuente academica
citable: 'Diagnostico sociocultural de diez municipios de la Sierra
Tarahumara', archivoshistoricosdeparral.com. VERIFICAR nombres exactos
contra tu catalogo antes de confiar en el merge (ver seccion 1).

In [ ]:
import pandas as pd

# Nombres tal como aparecen citados en la fuente academica (pueden no
# coincidir exactamente con la ortografia de tu catalogo CONAPO/INEGI)
SIERRA_TARAHUMARA_CITADOS = [
    'Balleza', 'Batopilas', 'Bocoyna', 'Carichí', 'Chínipas', 'Guachochi',
    'Guadalupe y Calvo', 'Guazapares', 'Guerrero', 'Maguarichi', 'Morelos',
    'Moris', 'Nonoava', 'Ocampo', 'Temósachic', 'Urique', 'Uruachi',
]
print(f'{len(SIERRA_TARAHUMARA_CITADOS)} municipios citados como Sierra Tarahumara')


## 1. Verificar que los nombres coincidan con tu catalogo (CRITICO)
Nombres oficiales INEGI a veces difieren (ej. 'Batopilas de Manuel Gomez
Morin' vs 'Batopilas'). Si un nombre citado no aparece en tu catalogo, el
merge lo va a perder silenciosamente -- por eso se verifica ANTES de usarlo.

In [ ]:
df_tasas = pd.read_csv('../data/processed/tasas_suicidio_municipal_chihuahua_2019_2024.csv', encoding='utf-8')
nombres_reales = set(df_tasas['NOM_MUN'].unique())

sin_match_exacto = [n for n in SIERRA_TARAHUMARA_CITADOS if n not in nombres_reales]
print('Nombres citados SIN match exacto en tu catalogo (revisar manualmente):')
for n in sin_match_exacto:
    # buscar el mas parecido para facilitar la correccion
    candidatos = [real for real in nombres_reales if n.split()[0].lower() in real.lower()]
    print(f'  "{n}" -> candidatos en tu catalogo: {candidatos}')


## 2. Corregir la lista con los nombres EXACTOS de tu catalogo
Ajusta este diccionario segun lo que haya mostrado la celda anterior antes
de continuar. Placeholder inicial basado en nombres INEGI conocidos --
VERIFICAR.

In [ ]:
# Mapeo de nombre citado -> nombre exacto en tu catalogo. Ajustar segun
# el resultado de la seccion 1.
CORRECCIONES_NOMBRE = {
    'Batopilas': 'Batopilas de Manuel Gómez Morín',
    'Temósachic': 'Temósachic',  # ajustar si tu catalogo usa 'Temósachi'
}

sierra_final = [CORRECCIONES_NOMBRE.get(n, n) for n in SIERRA_TARAHUMARA_CITADOS]
sierra_final_validado = [n for n in sierra_final if n in nombres_reales]
sierra_no_encontrado = [n for n in sierra_final if n not in nombres_reales]

print(f'Municipios de la Sierra Tarahumara validados en el catalogo: {len(sierra_final_validado)}/{len(SIERRA_TARAHUMARA_CITADOS)}')
if sierra_no_encontrado:
    print(f'AUN sin encontrar (corregir CORRECCIONES_NOMBRE): {sierra_no_encontrado}')


## 3. Comparar recurrencia: Sierra Tarahumara vs. resto del estado

In [ ]:
recurrencia = pd.read_csv('../data/processed/recurrencia_municipal_chihuahua.csv', encoding='utf-8')
recurrencia['es_sierra_tarahumara'] = recurrencia['NOM_MUN'].isin(sierra_final_validado)

resumen_sierra = recurrencia.groupby('es_sierra_tarahumara').agg(
    n_municipios=('NOM_MUN', 'count'),
    tasa_pooled_promedio=('tasa_pooled_100k', 'mean'),
    anios_en_top15_promedio=('anios_en_top15', 'mean'),
)
resumen_sierra


## 4. Autoadscripcion indigena (Conindig) en los CASOS de suicidio
Ver limitacion metodologica en la introduccion: esto es composicion de los
casos, NO tasa de la poblacion indigena.

In [ ]:
df_chih = pd.read_csv('../data/processed/suicidio_chihuahua_2019_2024.csv', encoding='utf-8', low_memory=False, dtype=str)

print('Distribucion de Conindig en TODOS los casos de Chihuahua:')
print(df_chih['Conindig'].value_counts(dropna=False))
print(f'(1=Si, 2=No, NaN=no especificado/se ignora tras recodificacion)')


In [ ]:
# Necesitamos el codigo de municipio (Mun_resid) de los municipios de la
# sierra, no el nombre -- lo sacamos del catalogo de poblacion ya cargado
df_pob_nombres = df_tasas[['mun_codigo', 'NOM_MUN']].drop_duplicates()
codigos_sierra = df_pob_nombres[df_pob_nombres['NOM_MUN'].isin(sierra_final_validado)]['mun_codigo'].tolist()

df_chih['es_sierra_tarahumara'] = df_chih['Mun_resid'].isin(codigos_sierra)

tabla_conindig = pd.crosstab(
    df_chih['es_sierra_tarahumara'], df_chih['Conindig'], dropna=False, normalize='index'
) * 100
print('% de casos por autoadscripcion indigena (Conindig), Sierra vs resto:')
tabla_conindig.round(1)


## 5. Hallazgos
_Documentar aqui: si el % de autoadscripcion indigena es notablemente mas
alto en los casos de la Sierra Tarahumara que en el resto del estado, y si
eso sostiene o matiza la hipotesis inicial. Recordar la limitacion de la
seccion 4: esto describe la composicion de los CASOS, no prueba causalidad
ni compara contra el % de poblacion indigena de cada region (dato que
requeriria una fuente adicional, ej. Censo de Poblacion y Vivienda)._